❗Note: I ran out of RAM due to One-Hot encoding so I am using this new notebook to transfer the strictly necessary commands and functions for this milestone ❗

# Exploring Netflix's TV Shows and Movies based on production details

Madeleinne Tan (ID 1221366627)

Goal: I was looking up datasets on Kaggle and I came across this: https://www.kaggle.com/datasets/shivamb/netflix-shows

I have always been interested in Netflix movies and TV shows, specifically with the following questions:
- What kinds of movies/shows are added and kept on Netflix?
- How often are movies/shows added to Netflix?
- What kinds of movies do Netflix recommend? Can we use clustering to look at the movies/shows Netflix have as of the creation of this dataset?

I'm especially interested in the last question. I would, eventually, like to do clustering to explore how the movies (or shows) are related on Netflix and how they keep items to attract new subscribers. Unfortunately, there are no datapoints about viewerships so it's hard to tell how subscribers react to movies.





List of features:
1. show_id : the unique ID for every movie / tv show
2. type : "movie" or "tv show"
3. title : title of the movie
4. director : director of the movie
5. cast : cast of the movie
6. country : where the movie or show was produced
7. date_added : the date it was added on Netflix
8. release_year : the actual release year of the movie or tv show.
9. rating : the TV rating of the movie or TV show (e.g. PG, TV-MA...)
10. duration : the total duration, in minutes (movies) or number of seasons (TV shows)
11. listed_in : the genre that the movie or tv show falls under (e.g. International TV Shows, Action, Documentary)
12. description: a summary description of the movie or TV show

Some important notes:
I have made changes directly onto the csv file while I was skimming through the data and I made some significant but helpful changes:
- Louis C.K's works were incorrectly inputted such that the rating feature contained duration data values, such as '86 minutes' instead of 'UR' for 'unrated'. I have realized this mistake when I was trying to make charts related to movie ratings.
- I made sure all of the dates were in Excel's short date format (e.g. 10/12/2023 for 'October 12, 2023').

I know these acts are technically supposed to be reserved for the cleaning, imputing, and data engineering sections, but I wanted to take care of these noticeable mistakes before they mess up the bar/count charts and other visualizations.

# Preparations

Note: this is skipping most of the important details that were once required for Milestone One. The only things here are the commands and functions that are relevant for Milestone 2

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

# Read the CSV file, make a dataframe
read_csv = pd.read_csv('netflix_titles.csv')
df = pd.DataFrame(read_csv)

# Note - I will only be making this about MOVIES so I can encode duration (the length of the movie) better. Movies also takes 70% of the dataset, so I have plenty of rows left.
df = df[df['type'] == 'Movie']
df['duration'] = df['duration'].str.replace(' min', '').astype(float)
df['duration'].describe()

ModuleNotFoundError: No module named 'numpy'

In [ ]:
df.shape

(6131, 12)

In [ ]:
# Edit: I'm going to try to delete a bunch of movies/rows just because I need as much RAM as possible when I do the One-Hot Encoding... sorry
# there are 6131 rows. I only need 500

# I'm going to remove 5500 rows because 6131 - 5500 = 631
chosen_indices = np.random.choice(df.index, 5500, replace=False)
df_subset = df.drop(chosen_indices)
df = df_subset

the reason why I'm removing so many columns is because the RAM keeps crashing for Milestones 2 and 3 being run in sequence... this also is necessary because the one-hot encoding with the cast list is BEEFY.

In [ ]:
df.shape

(631, 12)

In [ ]:
df.sample(10) # sample 10 random rows

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
1580,s1581,Movie,Spirit Riding Free: Ride Along Adventure,"Beth Sleven, Allan Jacobsen, Kevin Wotton","Amber Frank, Bailey Gambertoglio, Sydney Park,...",United States,12/8/2020,2020,TV-Y7,79.0,Children & Family Movies,Join Lucky and her friends on an interactive m...
2635,s2636,Movie,The Dealer,Ahmed Saleh,"Ahmed El Sakka, Khaled El Nabawy, Mai Selim, N...",Egypt,4/25/2020,2010,TV-14,124.0,"Action & Adventure, International Movies","Separated by distance and circumstance, two bi..."
7337,s7338,Movie,Lorai: Play to Live,Parambrata Chatterjee,"Prasenjit Chatterjee, Payel Sarkar, Indrasish ...",India,9/1/2017,2015,TV-14,155.0,"Dramas, International Movies, Sports Movies","Under a government initiative, a retired alcoh..."
1470,s1471,Movie,Alaska Is a Drag,Shaz Bennett,"Martin L. Washington Jr., Maya Washington, Mat...",United States,12/31/2020,2017,TV-MA,83.0,"Dramas, LGBTQ Movies","Tormented by bullies, an aspiring drag star wo..."
6473,s6474,Movie,Chitty Chitty Bang Bang,Ken Hughes,"Dick Van Dyke, Sally Ann Howes, Lionel Jeffrie...","United Kingdom, United States",1/1/2020,1968,G,146.0,"Children & Family Movies, Classic Movies, Come...",Quirky inventor Caractacus Potts and his famil...
2614,s2615,Movie,Boushkash,"Ahmed Yousry, Hazem Fouda","Mohamed Saad, Zeina, Ezzat Abou Aouf, Edward, ...",Egypt,4/28/2020,2008,TV-MA,100.0,"Comedies, International Movies",A former goalkeeper-turned-talent scout embark...
5950,s5951,Movie,Triumph of the Heart,Richard Michaels,"Mario Van Peebles, Susan Ruttan, Lane R. Davis...",United States,10/1/2011,1991,TV-PG,93.0,"Dramas, Sports Movies","This drama tells the tale of Ricky Bell, a pro..."
8168,s8169,Movie,Terra,"Yann Arthus-Bertrand, Michael Pitiot",Vanessa Paradis,France,5/1/2016,2015,TV-PG,98.0,"Documentaries, International Movies",This visually arresting documentary essay refl...
3476,s3477,Movie,The Squid and the Whale,Noah Baumbach,"Jeff Daniels, Laura Linney, Jesse Eisenberg, O...",United States,10/1/2019,2005,R,81.0,"Comedies, Dramas, Independent Movies",This insightful drama looks at the crumbling m...
4781,s4782,Movie,Luciano Mellera: Infantiloide,"Raúl Campos, Jan Suter",Luciano Mellera,Argentina,7/6/2018,2018,TV-14,66.0,Stand-Up Comedy,Argentina's Luciano Mellera emphasizes the hum...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 631 entries, 36 to 8761
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   show_id       631 non-null    object 
 1   type          631 non-null    object 
 2   title         631 non-null    object 
 3   director      614 non-null    object 
 4   cast          580 non-null    object 
 5   country       583 non-null    object 
 6   date_added    631 non-null    object 
 7   release_year  631 non-null    int64  
 8   rating        629 non-null    object 
 9   duration      631 non-null    float64
 10  listed_in     631 non-null    object 
 11  description   631 non-null    object 
dtypes: float64(1), int64(1), object(10)
memory usage: 64.1+ KB


In [ ]:
# Identifying outliers
import numpy as np
import pandas as pd
from scipy import stats

# calculate z-scores for continuous features
z_scores = df.select_dtypes(include=['float64', 'int64']).apply(stats.zscore).abs()

# identifying outlisers using z-score > 3
outliers_df = df[(z_scores > 3).any(axis=1)]

# print the num of outliers for each col
outliers_count = (z_scores > 3).sum()
print(f"Outliers detected based on Z-scores (threshold=3) per column:\n{outliers_count}")

# print the outliers in the form of a df
print("\nOutliers in the form of a dataframe:\n")
print(outliers_df)

Outliers detected based on Z-scores (threshold=3) per column:
release_year    18
duration         4
dtype: int64

Outliers in the form of a dataframe:

     show_id   type                                              title  \
2126   s2127  Movie                               What's Your Raashee?   
2543   s2544  Movie  Unbreakable Kimmy Schmidt: Kimmy vs. the Reverend   
2721   s2722  Movie                                             Duniya   
3142   s3143  Movie                                            Lakshya   
5220   s5221  Movie                                          Singapore   
5545   s5546  Movie                                              Elaan   
6473   s6474  Movie                            Chitty Chitty Bang Bang   
6746   s6747  Movie                                Fiddler on the Roof   
6765   s6766  Movie                               Five Elements Ninjas   
6784   s6785  Movie                                   Forbidden Planet   
6864   s6865  Movie               

In [ ]:
# Capping duration to be 180 minutes
df['duration'] = np.where(df['duration'] > 180, 180, df['duration'])
# Keeping the floor of the df for release_year to 1985
df['release_year'] = np.where(df['release_year'] < 1985, 1985, df['release_year'])

In [ ]:
# turning all TV-Y related ratings to just 'TV-Y' and merging 'NR' and 'UR' to just 'UR'

df['rating'] = df['rating'].replace(['TV-Y', 'TV-Y7', 'TV-Y7-FV'], 'TV-Y')
df['rating'] = df['rating'].replace(['NR', 'UR'], 'UR')
df['rating'].unique()
df = df.dropna() # this drops rows with any missing values; we're now down to 5188 entries and that's ok because to be honest, I'm a bit lazy and we have a lot of data we can afford to drop
df = df.drop_duplicates() # drop any duplicates

In [ ]:
# I just noticed TV-PG and PG are two different things so I will combine them. The same goes for TV-G and G
df['rating'] = df['rating'].replace(['TV-PG', 'PG'], 'PG')
df['rating'] = df['rating'].replace(['TV-G', 'G'], 'G')

In [ ]:
df['rating'].unique()

array(['R', 'TV-MA', 'PG', 'PG-13', 'TV-14', 'TV-Y', 'G', 'NC-17', 'UR'],
      dtype=object)

## Preparation for Encoding

## Modifying strings
I learned that a lot of names from the string-related features and values have non-English accents on the words, which throws JSON and the ML Model's compiler off.

In [ ]:
import re, unicodedata

# find columns with non-alphanumeric characters
pick_cols = [col for col in df.columns if re.search(r'[^a-zA-Z0-9_]', col)]
print("Problematic Columns:", pick_cols)



Problematic Columns: []


In [ ]:
import unicodedata

# Function to remove accents from a string
def remove_accents(input_str):
    return ''.join(
        c for c in unicodedata.normalize('NFKD', input_str)
        if not unicodedata.combining(c)
    )

# Apply to column names
df.columns = [remove_accents(col) for col in df.columns]

# Clean accents from the 'cast' column (values are lists)
df['cast'] = df['cast'].apply(
    lambda x: [remove_accents(name) for name in x] if isinstance(x, list) else x
)


In [ ]:
# Also, rename or fix the labeling of some of the names because some punctuations don't do well either with Lightgbm
df.columns = df.columns.str.replace('"{}[,\-\.\':]', '', regex=True).str.replace(' ', '_')

In [ ]:
import re, unicodedata

# find columns with non-alphanumeric characters
pick_cols = [col for col in df.columns if re.search(r'[^a-zA-Z0-9_]', col)]
print("Problematic Columns:", pick_cols)

Problematic Columns: []


In [ ]:
# Next, let's change up show_id so it represents the numerical value of the row in which the values are placed in
# I want to make sure all of the titles are alphabetically ordered first from A-Z

df = df.sort_values(by= 'title', ascending=True)

# numerically ordering the values and putting that as the show_id
df.reset_index(inplace = True, drop = True)
df['show_id'] = df.index

# ok, let's drop 'type' and 'title'
print(df.shape)
df.drop(columns=['type', 'title'], inplace=True)

(534, 12)


## Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

labelenc = LabelEncoder()

In [ ]:
# Encoding director
df['director'] = labelenc.fit_transform(df['director'])
df['director']

# Encoding country
df['country'] = labelenc.fit_transform(df['country'])
df['country']

# Encoding rating
rating_mapping = {
    'TV-Y': 0,
    'G' : 1,
    'PG' : 2,
    'PG-13': 3,
    'TV-14': 4,
    'TV-MA': 5,
    'R': 6,
    'NC-17': 7,
    'UR' : 8
}
df['rating'] = df['rating'].map(rating_mapping)

# Encoding dates
# Idea: turn date_added and release_year into 4 separate features:
# Feature 1: date_added_month
# Feature 2: date_added_day
# Feature 3: date_added_year
# Feature 4: release_year

df['date_added'] = pd.to_datetime(df['date_added'])
df['date_added_month'] = df['date_added'].dt.month
df['date_added_day'] = df['date_added'].dt.day
df['date_added_year'] = df['date_added'].dt.year

# I'm going to drop the original date_added
df.drop(columns=['date_added'], inplace=True)



## Normalization and Standardization

In [ ]:
# For dates
import numpy as np

# using cyclical feature engineering to normalize dated elements
# we only need it for months and days because they CYCLE - we don't need to do this for years
# and the reason why we do need both sine and cosine is because months (and days) may map to the same trig value if we're just using cosine or if we're just using sine. We need a sort of "coordinate pair" to represent the month

# for months
# df['date_added_month_cos'] = np.cos(2 * np.pi * df['date_added_month'] / 12)
# df['date_added_month_sin'] = np.sin(2 * np.pi * df['date_added_month'] / 12)

# # for days
# df['date_added_day_cos'] = np.cos(2 * np.pi * df['date_added_day'] / 30)
# df['date_added_day_sin'] = np.sin(2 * np.pi * df['date_added_day'] / 30)

# I have to rescale the values so that the outputs fall in [0,1] instead of [-1,1]
df['date_added_month_cos'] = (1 + np.cos(2 * np.pi * df['date_added_month'] / 12)) / 2
df['date_added_month_sin'] = (1 + np.sin(2 * np.pi * df['date_added_month'] / 12)) / 2

# for days
df['date_added_day_cos'] =(1+ np.cos(2 * np.pi * df['date_added_day'] / 30))/2
df['date_added_day_sin'] = (1+ np.sin(2 * np.pi * df['date_added_day'] / 30))/2

In [ ]:

# Years can be handled with regular standardiztion
# I will comment this out for now just because Naive Bayes doesn't like negative values (and the standard scaler is the cause of that)
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()

# df['release_year'] = scaler.fit_transform(df[['release_year']])
# df['date_added_year'] = scaler.fit_transform(df[['date_added_year']])


# Milestone 2

## Reflection before I do anything

What is the problem type?
- What immediately comes to mind are classification, clustering, and sentiment analysis
- Idea 1: Classify a movie's TV rating based on its description, director, duration, and cast?
  - Target column = rating; so this would be a supervised learning problem

- Idea 2: Cluster movies based on how similar they are in the release year, the tv rating, the director, the country of filming, and duration
  - No target variable, so this is unsupervised

I think I would like to do the first one (classification) because:
- It has the real-life application of predicting what a movie would be rated based on its content and who is involved in its creation
- At least in my mind, it's easier to measure the performance of the model when its purpose is to conclude discrete values








### Types of Machine Learning Models for Classification (pros and cons):
- Logistic Regression
  - Pros: very simple and interpretable, overfitting is preventable
  - Cons: binary answers only (ex: spam or not spam)
- K-Nearest Neighbors
  - Pros: classifies using closest neighbors (shares the advantages of clustering), non-linear decision boundaries can be modeled
  - Cons: sensitivity to outliers, dependent on how clusters and centroids are defined
- Decision Tree
  - Pros: no need for normalization
  - Cons: prone to overfitting, sensitive to small variations in data
- Random Forest
  - Pros: bagging (and boosting...?) helps it against overfitting
  - Cons: requires careful preparations beforehand
- Naive Bayes
  - Pros: simple and interpretable
  - Cons: assumes feature independence (which might be ok with this dataset?)
- Neural Networks
  - Pros: best ofr highly complex relationships and large scale data.
  - Cons: requires LARGE amounts of data
--------------------------------------
Based on these, my best bets are (1) KNN, (2) Naive Bayes

### My concerns
- How can I do ML on the arrays of strings (cast, listed_in, description)?

- Would description require a sort of sentiment analysis?

Note: I just added the new section in Milestone 1 titled "Other explorations". I just learned that none of the features in this dataset are significantly correlated, so I can totally do Naive Bayes classification because I computationally proved that since none of the features are correlated, then they exist independent of each other. Additionally, Naive Bayes seems simpler to do, as it doesn't require as much hyperparameter tuning as KNN.

## So, I choose Naive Bayes as my Machine Learning Model

In the slides, it also says "It's especially good for tasks like spam filtering or sentiment analysis, where features (like words) are somewhat independent"

Quick run through of how Naive Bayes (NB) works:
- Uses probability to make predictions; calculates the probability that a movie belongs to a rating based on its features using Bayes' Theorem
- TV rating with the highest probability is assigned to the new movie

## Preprocessing 'description', 'cast', and 'listed_in'

The reason why I didn't put this up in Milestone 1 was because (1) I didn't expect to be using Naive Bayes as my model of choice for until I started working on this milestone and (2) I wanted to make it clear that all of the work I did to preprocess description, cast, and listed_in were done for milestone 2

In [ ]:
df['description']

,description
0,"After a painful breakup, a trio of party-lovin..."
1,After one of his high school students attacks ...
2,"Seeking her independence, a young woman moves ..."
3,A bumbling Paris policeman is doggedly determi...
4,"In parallel love stories, the lives of college..."
...,...
529,"After a long stint in the army, an ex-lieutena..."
530,Two identical strangers pursue their respectiv...
531,Competing con artists attempt to creatively an...
532,Zack and Miri make and star in an adult film t...


In [ ]:
print(type(df['description']))
print(type(df['cast']))
print(type(df['listed_in']))

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>


In [ ]:
# Step 0: Import statements
from collections import defaultdict
import string
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
# Step 0.5 for 'description' --- lemmatization or stemming

import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize spaCy
nlp = spacy.load("en_core_web_sm")

# Lemmatization
def lemmatize_text(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])

# Implement/apply lemmatiztn
df['description'] = df['description'].apply(lemmatize_text)


In [ ]:
# Step 1: preprocess the text
stop_words = set(stopwords.words('english')) # stopwords are common words that may throw off the importance of frequency count of relevant words.
# stop words include things like 'the', 'and', 'a', ...

def preprocess(text):
  if isinstance(text, str):
    # process the string only if it's a valid str
    return [name.strip().lower() for name in text.split(',')]
  return []

def preprocess_desc(text):
    if isinstance(text, str):
        # process the string only if it's a valid str
        text = text.lower()  # we want all lowercase
        text = text.translate(str.maketrans('', '', '!"#$%&()*+,-./:;<=>?@[\]^_`{|}~'))  # Remove punctuation besides '
        tokens = text.split()  # tokenize into words
        tokens = [keep_word for keep_word in tokens if keep_word not in stop_words]
        return tokens

    elif isinstance(text, list):
        # (if the list has string elements, you might want to process those)
        tokens = [keep_word for keep_word in tokens if keep_word not in stop_words]
        return text

    else:
        # if it's neither a string nor a list, return an empty list
        return []

# preprocessing the different string vals
cast = [preprocess(cast) for cast in df['cast']]
description = [preprocess_desc(description) for description in df['description']]
listed_in = [preprocess(listed_in) for listed_in in df['listed_in']]



In [ ]:
print(cast)
print(description)
print(listed_in)

[['maia morgenstern', 'olimpia melinte', 'crina semciuc', 'flavia hojda', 'maria dinulescu', 'alex călin', 'vlad logigan', 'sali levent', 'silvia busuioc', 'mihaela mihut'], ['samuel l. jackson', 'john heard', 'kelly rowan', 'clifton collins jr.', 'tony plana'], ['nicole brydon bloom', 'giles matthey', 'taylor nichols', 'alan blumenfeld', 'celeste sully', 'susan davis', 'clayton hoff', 'earnestine phillips'], ['ramzy bedia', 'éric judor', 'benoît magimel', 'kristin scott thomas', 'élodie bouchez', 'édouard baer', 'fred testot', 'omar sy'], ['jacob elordi', 'adan canto', 'radha mitchell', 'tiera skovbye', 'kari matchett', 'tahmoh penikett'], ['anders danielsen lie', 'jon øigarden', 'jonas strand gravli', 'ola g. furuseth', 'maria bock', 'thorbjørn harr', 'jaden smith'], ['elle fanning', 'naomi watts', 'susan sarandon', 'tate donovan', 'linda emond', 'jordan carlos', 'sam trammell', 'maria dizzia', 'tessa albertson'], ['massimo ghini', 'ricky memphis', 'martina stella', 'ralph palka', 'b

In [ ]:
# Function to clean each string in the list or as a single string
import re

def clean_string(input_str):
    # Using re.sub for regex replacement
    cleaned_str = re.sub(r'[\"{}\[\],\-\.\':]', '', input_str)
    cleaned_str = cleaned_str.replace(' ', '_')  # Regular replace for spaces
    return cleaned_str


# Clean the 'cast' column, where each entry is a list of actor names
df['cast'] = df['cast'].apply(
    lambda x: [clean_string(name) for name in x] if isinstance(x, list) else clean_string(x)
)

In [ ]:
# Step 2: Build Vocabulary (for all three)
# Init a new empty set for vocab

cast_vocabulary = set()
desc_vocabulary = set()
listed_in_vocabulary = set()

for cast_member in cast:
  if cast_member: # this helps skip empty entries / null elements
    cast_vocabulary.update(cast_member)

for desc_word in description:
  if desc_word:
    desc_vocabulary.update(desc_word)

for listed_in_word in listed_in:
  if listed_in_word:
    listed_in_vocabulary.update(listed_in_word)

# Convert into sorted lists
cast_vocabulary = sorted(list(cast_vocabulary))
desc_vocabulary = sorted(list(desc_vocabulary))
listed_in_vocabulary = sorted(list(listed_in_vocabulary))

# let's see if it works...
print("Cast vocabulary: ", cast_vocabulary)



Cast vocabulary:  ['50 cent', 'a.j. cook', 'a.j. johnson', 'aaditi pohankar', 'aarohi patel', 'aaron eckhart', 'aaron eisenberg', 'aaron jeffery', 'aaron mccusker', 'aaron paul', 'aaron washington', 'aaron yoo', 'aarti patel', 'aarubala', 'abayomi alvin', 'abbas', 'abdalla mahmoud', 'abdel aziz el mountassir', 'abdel imam abdullah', 'abdullrahman al gohani', 'abhay deol', 'abhimanyu singh', 'abigail cruttenden', 'abigail pniowsky', 'abu valayamkulam', 'abun sungkar', 'acha septriasa', 'adam baldwin', 'adam beach', 'adam david thompson', 'adam faraizl', 'adam godley', 'adam gussow', 'adam hochstetter', 'adam lazarre-white', 'adam lefevre', 'adam pally', 'adam rose', 'adam sandler', 'adam tomei', 'adam tuominen', 'adan canto', 'adario strange', 'adarsh gourav', 'adebayo salami', 'adebukola', 'adebukola oladipupo', 'adeel akhtar', 'adel bencherif', 'adeline flaun', 'adewale akinnuoye-agbaje', 'adhisty zara', 'adhvik mahajan', 'adinia wirasti', 'aditi rao hydari', 'aditi vasudev', 'aditya 

In [ ]:
print("Description vocab: ", desc_vocabulary)

Description vocab:  ["'", "'s", '10', '100', '11', '1126', '12', '12th', '14', '15', '16', '17', '18th', '1900s', '1905', '1920s', '1930', '1958', '1960', '1970', '1980', '1981', '1983', '1990', '1992', '20', '2003', '2010', '2014', '2018', '2020', '2092', '20th', '21st', '25', '28', '3', '30', '34', '36', '40', '40th', '501', '51st', '60', '8', '80', '9', '90s', 'aardman', 'abandon', 'abbey', 'abduct', 'abductor', 'abel', 'ability', 'able', 'abominable', 'abortion', 'abound', 'abruptly', 'absence', 'abuse', 'academic', 'academy', 'acapulco', 'accept', 'acceptance', 'access', 'accident', 'accidentally', 'accompany', 'accomplice', 'accountant', 'accuse', 'acha', 'achieve', 'achmed', 'across', 'act', 'action', 'activist', 'activity', 'actor', 'actress', 'actually', 'adaptation', 'addict', 'aditi', 'adjust', 'admirer', 'admit', 'adore', 'adrien', 'adult', 'advanced', 'adventure', 'adventurous', 'affair', 'affection', 'affectionate', 'afghan', 'afghanistan', 'afloat', 'africa', 'african', 

In [ ]:
# I'm curious about their sizes...
print(f"Cast Vocabulary Size: {len(cast_vocabulary)}")
print(f"Description Vocabulary Size: {len(desc_vocabulary)}")
print(f"Listed_in Vocabulary Size: {len(listed_in_vocabulary)}")

# the number of unique vocab words looks right...

Cast Vocabulary Size: 3848
Description Vocabulary Size: 3332
Listed_in Vocabulary Size: 20


In [ ]:
print("Listed_In vocab: ", listed_in_vocabulary)
# I want to remove 'movies' from the vocabulary and the listed_in options... but I'm not sure whether it's worth it
# How many movies have 'movie' in it?
print(listed_in)
print(listed_in.count(['movies']))

# ok so there are rows with just 'movies' in their 'listed_in' so I just want to keep them for now... for peace of mind...

Listed_In vocab:  ['action & adventure', 'anime features', 'children & family movies', 'classic movies', 'comedies', 'cult movies', 'documentaries', 'dramas', 'faith & spirituality', 'horror movies', 'independent movies', 'international movies', 'lgbtq movies', 'movies', 'music & musicals', 'romantic movies', 'sci-fi & fantasy', 'sports movies', 'stand-up comedy', 'thrillers']
[['comedies', 'dramas', 'international movies'], ['dramas'], ['horror movies', 'independent movies', 'thrillers'], ['comedies', 'international movies'], ['dramas', 'faith & spirituality', 'romantic movies'], ['dramas', 'thrillers'], ['dramas', 'lgbtq movies'], ['comedies', 'international movies'], ['comedies', 'dramas', 'romantic movies'], ['comedies', 'romantic movies'], ['dramas', 'international movies', 'music & musicals'], ['comedies'], ['movies'], ['children & family movies', 'comedies', 'music & musicals'], ['comedies'], ['documentaries'], ['horror movies', 'independent movies'], ['dramas', 'thrillers'], ['

## Encoding the "Big 3" ('cast', 'listed_in', and 'description)
Reasoning: Naive Bayes uses numbers to predict, so we need to have numerical representations for each of these features...

In [ ]:
# I want to put the lists into df
df['cast'] = cast
df['description'] = description
df['listed_in'] = listed_in
df.sample(5)

,show_id,director,cast,country,release_year,rating,duration,listed_in,description,date_added_month,date_added_day,date_added_year,date_added_month_cos,date_added_month_sin,date_added_day_cos,date_added_day_sin
333,333,38,"[sanjay dutt, arjun kapoor, kriti sanon, zeena...",34,2019,4,171.0,"[dramas, international movies, romantic movies]","[18th, century, india, maratha, commander, lea...",2,14,2020,0.75,0.933013,0.010926,0.603956
460,460,124,[rachael stirling],78,2017,2,44.0,[documentaries],"[jessica, diana, sister, separate, year, age, ...",6,1,2017,0.00,0.500000,0.989074,0.603956
97,97,222,"[chiwetel ejiofor, martin sheen, danny glover,...",89,2018,4,105.0,"[dramas, faith & spirituality, independent mov...","[crisis, faith, set, renowned, fundamentalist,...",4,13,2018,0.25,0.933013,0.043227,0.703368
236,236,254,"[ezgi mola, murat yıldırım, gülenay kalkan, eb...",75,2015,5,104.0,"[comedies, international movies, romantic movies]","[30, year, old, never, take, traditional, path...",4,23,2021,0.25,0.933013,0.552264,0.002739
256,256,25,"[vicky kaushal, angira dhar, alankrita sahai, ...",34,2018,4,133.0,"[comedies, international movies, romantic movies]","[individually, bank, employee, sanjay, karina,...",2,14,2018,0.75,0.933013,0.010926,0.603956


About Encoding:

'Cast' and 'Listed_in' are categorical features (because they categorize movies) but 'description' is a text feature (they describe, but not categorize movies).

So, for categorical variables, we can do One-Hot Encoding. We don't want to do label encoding because label encoding can imply an order or 'ranking' nature to the values, which is untrue.

For 'description', we need to do either CountVectorizer (which values the words by frequency), or TF-IDF (which puts values on the words by relevance. This is good for making sure keywords like 'thriller' in the description mean more than 'movie' or 'person' showing up in the description)

In [ ]:
# ONE HOT ENCODING FOR CAST AND LISTED_IN
# https://www.geeksforgeeks.org/one-hot-encoding-from-a-pandas-column-containing-a-list/

import pandas as pd
import numpy as np

# (1) flatten lists and get unique categories
# flatten cast
all_cast_members = list([person for sublist in df['cast'] for person in sublist])
all_genres = list([genre for sublist in df['listed_in'] for genre in sublist])

# (2) create one hot encoded columns
# One-hot encoding for 'cast'
df_cast_encoded = pd.DataFrame(
    [[1 if person in cast else 0 for person in all_cast_members] for cast in df['cast']],
    columns=all_cast_members
)

# One-hot encoding for 'listed_in'
df_listed_in_encoded = pd.DataFrame(
    [[1 if genre in listed else 0 for genre in all_genres] for listed in df['listed_in']],
    columns=all_genres
)


In [ ]:
# # bring it back to original DataFrame
# df_combined = pd.concat([df, df_cast_encoded, df_listed_in_encoded], axis=1)

# # drop the original 'cast' and 'listed_in' columns if you don't need them anymore
# df_combined = df_combined.drop(columns=['cast', 'listed_in'])

# df_combined.sample(5)

In [ ]:
df_cast_encoded.sample(5)

,maia morgenstern,olimpia melinte,crina semciuc,flavia hojda,maria dinulescu,alex călin,vlad logigan,sali levent,silvia busuioc,mihaela mihut,...,jonah bobo,josh hutcherson,dax shepard,kristen stewart,tim robbins,frank oz,john alexander,derek mears,douglas tait,joe bucaro iii
149,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
59,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
490,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
120,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
56,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df['description'] = df['description'].apply(lambda x: ' '.join(x))
df['description'].sample(3)

,description
429,matriarch bent lavish 51st birthday party fami...
36,monster go mouse nap spree new york fievel fri...
421,grow post apocalyptic 2018 john connor must le...


In [ ]:
# TF-IDF FOR DESCRIPTION
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()
desc_encoded = vectorizer.fit_transform(df['description'])

In [ ]:
# Convert sparse matrix to dense array and create DataFrame
desc_encoded_dense = desc_encoded.toarray()  # we want to convert it to dense array
df_desc_encoded = pd.DataFrame(desc_encoded_dense, columns=vectorizer.get_feature_names_out())

# concatenate with other features
df_combined = pd.concat([df_cast_encoded, df_listed_in_encoded, df_desc_encoded], axis=1)

In [ ]:
df_combined.sample(5)

,maia morgenstern,olimpia melinte,crina semciuc,flavia hojda,maria dinulescu,alex călin,vlad logigan,sali levent,silvia busuioc,mihaela mihut,...,young,youth,youtube,yuletide,zack,zany,zathura,zealot,zombie,şeref
476,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
137,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
312,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
62,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
189,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df = pd.concat([df, df_combined], axis=1)
df.sample(5)

,show_id,director,cast,country,release_year,rating,duration,listed_in,description,date_added_month,...,young,youth,youtube,yuletide,zack,zany,zathura,zealot,zombie,şeref
425,425,155,"[sam ashe arnold, jakob davies, dalila bela, r...",10,2016,0,88.0,[children & family movies],find peculiar key three smart adventurous kid ...,5,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
287,287,449,"[julia roberts, lily collins, armie hammer, na...",93,2012,2,106.0,"[children & family movies, comedies]",remake classic grimm tale follow fair skinned ...,4,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
135,135,309,[david a. arnold],89,2020,5,61.0,[stand-up comedy],finally comfortable skin season comic david ar...,3,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
382,382,480,"[david arquette, neve campbell, courteney cox,...",89,2000,6,117.0,[horror movies],installment tongue cheek horror franchise find...,7,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
76,76,193,"[brad paisley, nate bargatze, john heffron, jo...",89,2017,5,63.0,"[music & musicals, stand-up comedy]",country music star brad paisley host night mus...,8,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


No way I have over 15k columns just from one-hot encoding cast members and descriptions alone 😢

In [ ]:
# Ok I need to drop the original text-related columns of the Big 3 before passing it into a Naive Bayes model
df = df.drop(columns=['cast', 'listed_in', 'description'])

In [ ]:
df.sample(5)

,show_id,director,country,release_year,rating,duration,date_added_month,date_added_day,date_added_year,date_added_month_cos,...,young,youth,youtube,yuletide,zack,zany,zathura,zealot,zombie,şeref
412,412,477,10,2014,7,112.0,9,24,2017,0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
435,435,427,3,2015,5,95.0,4,26,2017,0.250000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
321,321,272,3,2018,6,120.0,1,3,2019,0.933013,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
508,508,470,34,2019,4,116.0,8,2,2019,0.250000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
531,531,99,68,2018,5,130.0,2,15,2019,0.750000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Remove duplicate cast name features (I found this out when doing Milestone 3 work)
# solution found on https://stackoverflow.com/questions/14984119/python-pandas-remove-duplicate-columns
df = df.loc[:,~df.columns.duplicated()].copy()

## Split Data

In [ ]:
# split data (70/30)
from sklearn.model_selection import train_test_split

# define X and y
X = df.drop(columns=['rating']) # everything is 'X' except the rating
y = df['rating']

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Naive Bayes Model

In [ ]:
# Filter rows with any negative values
negative_rows = df[(df < 0).any(axis=1)]

print(negative_rows)

Empty DataFrame
Columns: [show_id, director, country, release_year, rating, duration, date_added_month, date_added_day, date_added_year, date_added_month_cos, date_added_month_sin, date_added_day_cos, date_added_day_sin, maia morgenstern, olimpia melinte, crina semciuc, flavia hojda, maria dinulescu, alex călin, vlad logigan, sali levent, silvia busuioc, mihaela mihut, samuel l. jackson, john heard, kelly rowan, clifton collins jr., tony plana, nicole brydon bloom, giles matthey, taylor nichols, alan blumenfeld, celeste sully, susan davis, clayton hoff, earnestine phillips, ramzy bedia, éric judor, benoît magimel, kristin scott thomas, élodie bouchez, édouard baer, fred testot, omar sy, jacob elordi, adan canto, radha mitchell, tiera skovbye, kari matchett, tahmoh penikett, anders danielsen lie, jon øigarden, jonas strand gravli, ola g. furuseth, maria bock, thorbjørn harr, jaden smith, elle fanning, naomi watts, susan sarandon, tate donovan, linda emond, jordan carlos, sam trammell, m

In [ ]:
# https://www.datacamp.com/tutorial/naive-bayes-scikit-learn
from sklearn.naive_bayes import MultinomialNB

nb = MultinomialNB() # init NB model

# fit the model
nb.fit(X_train, y_train)

# make prediction about the rating
y_predict = nb.predict(X_test)

# taking an example
# print("Actual val: ", y_test[0]) # wait this didn't work
# print("Predicted : ", y_predict[0])

## Model Evaluation

remember:
- false positive = you think it's positive, but it isn't

- false negative = you think it's negative, but it isn't

Reminder:
- Precision = TP / (TP+FP)
- Recall = TP / (TP+FN)
- F1 = 2*((Precision x Recall) / (Precision+Recall))
- Support = actual num of smaples per category

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# find the model's accuracy
accuracy = accuracy_score(y_test, y_predict)
print('Accuracy: ', accuracy)

# precision
print('Precision', precision_score(y_test, y_predict, average='weighted'))

# recall
print('Recall', recall_score(y_test, y_predict, average='weighted'))

# f1
print('F1', f1_score(y_test, y_predict, average='weighted'))

# support
# print('Support', support_score(y_test, y_predict, average='weighted'))
print("\n",classification_report(y_test, y_predict))

Accuracy:  0.29906542056074764
Precision 0.21170449488206497
Recall 0.29906542056074764
F1 0.14948057856723096

               precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00        19
           3       0.00      0.00      0.00         9
           4       0.50      0.04      0.07        27
           5       0.30      1.00      0.46        31
           6       0.00      0.00      0.00        17

    accuracy                           0.30       107
   macro avg       0.11      0.15      0.07       107
weighted avg       0.21      0.30      0.15       107



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

In [ ]:
# Using a confusion matrix to visualize the results, especially what are the True Positives/True Negatives
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import metrics

# referring to zybooks' classification metrics unit
# calculate the confusion matrix for each model
print("Naive Bayes model \n", metrics.confusion_matrix(y_test, y_predict))


Naive Bayes model 
 [[ 0  0  0  0  0  2  0]
 [ 0  0  0  0  0  2  0]
 [ 0  0  0  0  0 19  0]
 [ 0  0  0  0  0  9  0]
 [ 0  0  0  0  1 26  0]
 [ 0  0  0  0  0 31  0]
 [ 0  0  0  0  1 16  0]]


Interpreting the confusion matrix: Rows represent the actual classes (true labels), columns are the predicted classes. So, cm[n][n] represents a correct and true prediction.

In [ ]:
df['rating'].unique()

array([5, 6, 3, 4, 0, 2, 1, 8, 7])

Recall that this is the code for encoding the ratings:

```
# Encoding rating
rating_mapping = {
    'TV-Y': 0,
    'G' : 1,
    'PG' : 2,
    'PG-13': 3,
    'TV-14': 4,
    'TV-MA': 5,
    'R': 6,
    'NC-17': 7,
    'UR' : 8
}
df['rating'] = df['rating'].map(rating_mapping)
```
Since there are 8/9 of the ratings that showed up, the matrix is 8x8


EDIT: Sometimes the confusion matrix changes dimensions because I randomly deleted a LOT of rows, so it would not be a surprise to know that a lot of the ratings in one rating category disappeared.

## Reflections

- Reflection: this model has a very low accuracy (42.27%) and a poor F1 score.
  - I've been referring to https://encord.com/blog/f1-score-in-machine-learning/#:~:text=How%20do%20you%20interpret%20the,to%20have%20a%20poor%20performance. to figure out how to interpret F1

- Honestly, I'm not surprised. There's a lot that goes into a movie to determine its rating. For example, in this dataset, we don't consider profanity, non-family-friendly content, and thematic elements. But I also really wonder what could better predict the movie ratings... Am I using too few models? Should I have gone for a Random Forest situation where I use multiple learning models and combine them? Maybe if I had a little more time, that would be something I can explore...

# Post-Milestone 2
This includes all things that I should have completed in Milestone 2 but didn't get to do (or forgot to do) back then.

In [ ]:
# First, I wonder if I overfitted or underfitted the model
# I will check by testing the model against the training data

y_predict_ontraining = nb.predict(X_train)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# find the model's accuracy
accuracy = accuracy_score(y_test, y_predict)
print('Accuracy: ', accuracy)

# precision
print('Precision', precision_score(y_test, y_predict, average='weighted'))

# recall
print('Recall', recall_score(y_test, y_predict, average='weighted'))

# f1
print('F1', f1_score(y_test, y_predict, average='weighted'))

# support
# print('Support', support_score(y_test, y_predict, average='weighted'))
print("\n",classification_report(y_test, y_predict))

Accuracy:  0.29906542056074764
Precision 0.21170449488206497
Recall 0.29906542056074764
F1 0.14948057856723096

               precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00        19
           3       0.00      0.00      0.00         9
           4       0.50      0.04      0.07        27
           5       0.30      1.00      0.46        31
           6       0.00      0.00      0.00        17

    accuracy                           0.30       107
   macro avg       0.11      0.15      0.07       107
weighted avg       0.21      0.30      0.15       107



/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/m

So, it looks like I actually ***underfitted*** the model, because it's doing poorly in both testing and training data in terms of accuracy. They're both about 43%. I wonder if it's because I encoded the words in description incorrectly.

Edit: I implemented lemmatization before TF-IDF encoding for description and it increased accuracies from 43% to 45%. I think it's interesting because I wished it could have gone higher, but at the same time, I understand that this is a complex dataset.

## Cross Validating the Naive Bayes Model

I will choose another model to run and test the data on. My best options at the moment are:
- Gradient Boosting
  - High accuracy
  - effective at handling noisy datasets
  - Computationally intensive, prone to overfitting
- Random Forest
  - multiple decision trees
  - allows bagging and ensemble learning
  - reduces overfitting compared to a single dec tree
  - slower to train, may be less interpretable
  - requires careful fine tuning
- NN Models



In [ ]:
y.unique()

array([5, 6, 3, 4, 0, 2, 1, 8, 7])

In [ ]:
df.sample(5)

,show_id,director,country,release_year,rating,duration,date_added_month,date_added_day,date_added_year,date_added_month_cos,...,young,youth,youtube,yuletide,zack,zany,zathura,zealot,zombie,şeref
406,406,303,89,2006,4,94.0,9,15,2019,0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
451,451,468,89,2020,3,110.0,8,10,2020,0.250000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
351,351,46,89,1988,6,134.0,3,1,2021,0.500000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
119,119,134,38,2019,4,118.0,5,14,2020,0.066987,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
346,346,408,34,2019,4,132.0,4,29,2020,0.250000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# pip install dask[dataframe]

In [ ]:
# export df to csv
df.to_csv('test.csv', index=False)

In [ ]:
df.sample()

,show_id,director,country,release_year,rating,duration,date_added_month,date_added_day,date_added_year,date_added_month_cos,...,young,youth,youtube,yuletide,zack,zany,zathura,zealot,zombie,şeref
524,524,64,89,2020,5,88.0,4,4,2021,0.25,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
pick_cols = [col for col in df.columns if re.search(r'[^a-zA-Z0-9_]', col)]
print("Problematic Columns:", pick_cols)

Problematic Columns: ['maia morgenstern', 'olimpia melinte', 'crina semciuc', 'flavia hojda', 'maria dinulescu', 'alex călin', 'vlad logigan', 'sali levent', 'silvia busuioc', 'mihaela mihut', 'samuel l. jackson', 'john heard', 'kelly rowan', 'clifton collins jr.', 'tony plana', 'nicole brydon bloom', 'giles matthey', 'taylor nichols', 'alan blumenfeld', 'celeste sully', 'susan davis', 'clayton hoff', 'earnestine phillips', 'ramzy bedia', 'éric judor', 'benoît magimel', 'kristin scott thomas', 'élodie bouchez', 'édouard baer', 'fred testot', 'omar sy', 'jacob elordi', 'adan canto', 'radha mitchell', 'tiera skovbye', 'kari matchett', 'tahmoh penikett', 'anders danielsen lie', 'jon øigarden', 'jonas strand gravli', 'ola g. furuseth', 'maria bock', 'thorbjørn harr', 'jaden smith', 'elle fanning', 'naomi watts', 'susan sarandon', 'tate donovan', 'linda emond', 'jordan carlos', 'sam trammell', 'maria dizzia', 'tessa albertson', 'massimo ghini', 'ricky memphis', 'martina stella', 'ralph palk

## Lightgbm Gradient Boosting

In [ ]:
# # I'm going to go with Gradient Boosting with Light GBM because I learned that Light GBM does decently well with large datasets with categorical support

# import lightgbm as light

# # initialize model
# light_model = light.LGBMClassifier(
#     objective='multiclass',
#     num_class = len(y.unique())
# )

# # fit the model
# light_model.fit(X_train, y_train)

# # make prediction
# y_predict_light = light_model.predict(X_test)


In [ ]:
# # Evaluating the lightgbm model
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# # find the model's accuracy
# accuracy = accuracy_score(y_test, y_predict)
# print('Accuracy: ', accuracy)

# # precision
# print('Precision', precision_score(y_test, y_predict, average='weighted'))

# # recall
# print('Recall', recall_score(y_test, y_predict, average='weighted'))

# # f1
# print('F1', f1_score(y_test, y_predict, average='weighted'))

# # support
# # print('Support', support_score(y_test, y_predict, average='weighted'))
# print("\n",classification_report(y_test, y_predict))

# Milestone 3



Alright so I have to preface this with saying:

I attempted to cross validate my model and do some parameter fine-tuning before working on this milestone, but I ran into a lot of roadblocks such as inserting steps and debugging functions that belong in Milestones 1 and 2; I didn't anticipate the yarn that I feel like I couldn't finish unraveling. So, I will do milestone 3 to the best of my abilities. Thank you for your patience :D

In [ ]:
df.sample(5)

,show_id,director,country,release_year,rating,duration,date_added_month,date_added_day,date_added_year,date_added_month_cos,...,young,youth,youtube,yuletide,zack,zany,zathura,zealot,zombie,şeref
279,279,100,89,2013,4,84.0,1,15,2017,0.933013,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
307,307,121,89,2018,4,113.0,7,1,2019,0.066987,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
162,162,327,53,2017,4,93.0,1,31,2020,0.933013,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
116,116,490,24,1997,4,136.0,6,19,2020,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
215,215,57,89,2018,5,101.0,4,16,2019,0.250000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 534 entries, 0 to 533
Columns: 7197 entries, show_id to şeref
dtypes: float64(3321), int32(3), int64(3873)
memory usage: 29.3 MB


## Final Model Performance Analysis

I'm going to use SHAP values and LIME to get insights about the Naive Bayes Model.

### SHAP Values

Following the tutorial here: https://www.datacamp.com/tutorial/introduction-to-shap-values-machine-learning-interpretability

In [ ]:
pip install shap

In [ ]:
import shap
shap.initjs()

In [ ]:
# explainer = shap.Explainer(nb, X_train)
# shap_values = explainer(X_test)
# the above code got some errors so I'm trying this solution:

# use a function to wrap the MultinomialNB's predict_proba() for SHAP
def model_predict_proba(X):
    return nb.predict_proba(X)

# Now pass this wrapped function to the Explainer
explainer = shap.Explainer(model_predict_proba, X_train, max_evals=((2*len(df.columns))+1))  # it needs to be higher than 2*num_features + 1
shap_values = explainer(X_test)

# NOOO THE RAM CRASHED - I went from deleting 5000 rows to 5500 rows in the start of this program for this reason... let's see if this works

Alright. I'm moving to Visual Studio


## Deployment Plan

## Ethical Considerations
